In [1]:
from datasets import load_dataset, concatenate_datasets

/opt/anaconda3/envs/ragenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# importing RagBench data set 
domain_legal_list = ['cuad']
domain_custsupp_list = ['delusionqa', 'emanual', 'techqa']
domain_fin_list = ['finqa', 'tatqa']
domain_bio_list = ['pubmedqa', 'covidqa']
domain_gen_list = ['hotpotqa', 'msmarco', 'hagrid', 'expertqa']

In [3]:
def load_ragbench_subsets(
    subsets: list[str],
    split: str = "test",
    add_dataset_name: bool = True,
):
    """
    Load selected RAGBench subsets and concatenate them.

    Args:
        subsets: List of RAGBench subset names.
        split: Dataset split to load (default: test).
        add_dataset_name: Add source dataset name column.

    Returns:
        Combined Hugging Face Dataset.
    """

    datasets_list = []

    for subset in subsets:
        print(f"Loading subset: {subset}")

        ds = load_dataset(
            "rungalileo/ragbench",
            subset,
            split=split
        )

        if add_dataset_name:
            ds = ds.add_column(
                "dataset_name",
                [subset] * len(ds)
            )

        datasets_list.append(ds)

    combined_dataset = concatenate_datasets(datasets_list)

    print(f"Total records: {len(combined_dataset)}")

    return combined_dataset

In [91]:
legal_data = load_ragbench_subsets(subsets=["pubmedqa"], split="test", add_dataset_name=False)

Loading subset: pubmedqa
Total records: 2450


In [96]:
legal_data['documents'][10]

['To evaluate the influence of prognostic factors in postoperative radiotherapy of NSCLC with special emphasis on the time interval between surgery and start of radiotherapy.',
 'In univariate analysis, Karnofsky performance status (90+>60-80%; p = 0.019 log rank), resection status stratified for nodal disease (R+<R0; p = 0.046), and the time interval between SX and RT were of significant importance. Patients with a long interval (37 to 84 days) had higher 5-year survival rates (26%) and a median survival time (MST: 21.9 months, 95% C.I. 17.2 to 28.6 months) than patients with a short interval (18 to 36 days: 15%; 14.9 months, 13 to 19.9 months; p = 0.013). A further subgroup analysis revealed significant higher survival rates in patients with a long interval in N0/1 disease (p = 0.011) and incompletely resected NSCLC (p = 0.012). In multivariate analysis, the time interval had a p-value of 0.009 (nodal disease: p = 0.0083; KPI: p = 0.0037; sex: p = 0.035).',
 'The optimal interval bet

In [92]:
from langchain_core.documents import Document

raw_documents = []
seen_docs = set()

# for i in range(min(5, len(legal_data))):  # Indexing a small subset for demonstration
for doc_text in legal_data['documents']:
    for docs in doc_text:
        if docs not in seen_docs:
            seen_docs.add(docs)
            # Wrap standard strings into LangChain Document objects
            raw_documents.append(docs)


print(len(raw_documents))

# print(f"Ingesting {len(raw_documents)} unique documents into the Vector Store...")

5932


In [93]:
raw_documents

['(1) Can surface porous PEEK (PEEK-SP) microstructure be reliably controlled? (2) What is the effect of pore size on the mechanical properties of PEEK-SP? (3) Do surface porosity and pore size influence the cellular response to PEEK?',
 'Micro-CT analysis showed that PEEK-SP layers possessed pores that were 284\xa0±\xa035\xa0µm, 341\xa0±\xa049\xa0µm, and 416\xa0±\xa054\xa0µm for each pore size group. Porosity and pore layer depth ranged from 61% to 69% and 303 to 391\xa0µm, respectively. Mechanical testing revealed tensile strengths\xa0>\xa067\xa0MPa and interfacial shear strengths\xa0>\xa020\xa0MPa for all three pore size groups. All PEEK-SP groups exhibited\xa0>\xa050% decrease in ductility compared with PEEK-IM and demonstrated fatigue strength\xa0>\xa038\xa0MPa at one million cycles. All PEEK-SP groups also supported greater proliferation and cell-mediated mineralization compared with smooth PEEK and Ti6Al4V.',
 "Despite its widespread use in orthopaedic implants such as soft tiss

In [15]:
legal_data

Dataset({
    features: ['id', 'question', 'documents', 'response', 'generation_model_name', 'annotating_model_name', 'dataset_name', 'documents_sentences', 'response_sentences', 'sentence_support_information', 'unsupported_response_sentence_keys', 'adherence_score', 'overall_supported_explanation', 'relevance_explanation', 'all_relevant_sentence_keys', 'all_utilized_sentence_keys', 'trulens_groundedness', 'trulens_context_relevance', 'ragas_faithfulness', 'ragas_context_relevance', 'gpt3_adherence', 'gpt3_context_relevance', 'gpt35_utilization', 'relevance_score', 'utilization_score', 'completeness_score'],
    num_rows: 510
})

In [65]:
# # preprocessing the documents data which is text and remove the new line characters greater than '\n' from the documents data
# def preprocess_documents(documents):
#     preprocessed_docs = []
#     for doc in documents:
#         # Remove new line characters
#         # Replace multiple new lines with a single one
#         cleaned_doc = doc.replace('\n\n', '\n') 
#         preprocessed_docs.append(cleaned_doc)

#     return '\n'.join(preprocessed_docs)

# preprocessing the documents data which is text and remove the new line characters greater than '\n' from the documents data
def preprocess_documents(documents):
    return documents.replace('\n\n', '\n') 
    

In [58]:
legal_data['documents'][9]


['EXHIBIT 10.1   RESELLER AGREEMENT   THIS RESELLER AGREEMENT (this "Agreement") is made and entered into effect the 7th day of April, 2017 ("Effective Date"), by and between i3 Integrative Creative Solutions, LLC ("i3 ICS"), a Virginia limited liability company, having its offices at 6564 Loisdale Court Suite 1010B, Springfield, VA 22150 ("Reseller") and the company set forth below ("Company") (each, individually, a "party" and collectively, "parties"):   Company: Bravatek Solutions, Inc. (BVTK) Telephone: 1-866-490-8590\n\nAddress:2028 E. Ben White Blvd., Suite 240-2835 Fax: N/A\n\nAustin, Texas 78741 E-mail: tom.cellucci@bravatek.com\n\nTerritory: US Federal Government Civilian and Military Agencies/Customers in the U.S. Agreement Term: 1 Year\n\nCompany Products: cybersecurity email software/telecom services Other Terms (not applicable if blank):\n\nPricing: Reseller will obtain pricing quote from Company for each opportunity. Contract is renewable for 1 year extension by amendment

In [79]:
print(preprocess_documents(legal_data['documents'][9]))

EXHIBIT 10.1   RESELLER AGREEMENT   THIS RESELLER AGREEMENT (this "Agreement") is made and entered into effect the 7th day of April, 2017 ("Effective Date"), by and between i3 Integrative Creative Solutions, LLC ("i3 ICS"), a Virginia limited liability company, having its offices at 6564 Loisdale Court Suite 1010B, Springfield, VA 22150 ("Reseller") and the company set forth below ("Company") (each, individually, a "party" and collectively, "parties"):   Company: Bravatek Solutions, Inc. (BVTK) Telephone: 1-866-490-8590
Address:2028 E. Ben White Blvd., Suite 240-2835 Fax: N/A
Austin, Texas 78741 E-mail: tom.cellucci@bravatek.com
Territory: US Federal Government Civilian and Military Agencies/Customers in the U.S. Agreement Term: 1 Year
Company Products: cybersecurity email software/telecom services Other Terms (not applicable if blank):
Pricing: Reseller will obtain pricing quote from Company for each opportunity. Contract is renewable for 1 year extension by amendment to this agreemen

In [55]:
raw_documents[9]


'EXHIBIT 10.1   RESELLER AGREEMENT   THIS RESELLER AGREEMENT (this "Agreement") is made and entered into effect the 7th day of April, 2017 ("Effective Date"), by and between i3 Integrative Creative Solutions, LLC ("i3 ICS"), a Virginia limited liability company, having its offices at 6564 Loisdale Court Suite 1010B, Springfield, VA 22150 ("Reseller") and the company set forth below ("Company") (each, individually, a "party" and collectively, "parties"):   Company: Bravatek Solutions, Inc. (BVTK) Telephone: 1-866-490-8590\n\nAddress:2028 E. Ben White Blvd., Suite 240-2835 Fax: N/A\n\nAustin, Texas 78741 E-mail: tom.cellucci@bravatek.com\n\nTerritory: US Federal Government Civilian and Military Agencies/Customers in the U.S. Agreement Term: 1 Year\n\nCompany Products: cybersecurity email software/telecom services Other Terms (not applicable if blank):\n\nPricing: Reseller will obtain pricing quote from Company for each opportunity. Contract is renewable for 1 year extension by amendment 

In [66]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
# texts = text_splitter.split_text(preprocess_documents(legal_data['documents'][9]))
texts = text_splitter.split_text(preprocess_documents(raw_documents[9]))

In [71]:
len(preprocess_documents(raw_documents[9]))

15832

In [67]:
len(texts)

27

In [73]:
texts[0]

'EXHIBIT 10.1   RESELLER AGREEMENT   THIS RESELLER AGREEMENT (this "Agreement") is made and entered into effect the 7th day of April, 2017 ("Effective Date"), by and between i3 Integrative Creative Solutions, LLC ("i3 ICS"), a Virginia limited liability company, having its offices at 6564 Loisdale Court Suite 1010B, Springfield, VA 22150 ("Reseller") and the company set forth below ("Company") (each, individually, a "party" and collectively, "parties"):   Company: Bravatek Solutions, Inc. (BVTK) Telephone: 1-866-490-8590\nAddress:2028 E. Ben White Blvd., Suite 240-2835 Fax: N/A\nAustin, Texas 78741 E-mail: tom.cellucci@bravatek.com\nTerritory: US Federal Government Civilian and Military Agencies/Customers in the U.S. Agreement Term: 1 Year\nCompany Products: cybersecurity email software/telecom services Other Terms (not applicable if blank):'

In [ ]:
# import chromadb
# from chromadb.utils import embedding_functions

# emberding_function = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="BAAI/bge-small-en-v1.5") #"all-MiniLM-L6-v2"

# client = chromadb.PersistentClient(path="./chroma_db1")
# collection = client.get_or_create_collection(name="test",
#                                         metadata={"description": "Collection of legal documents from RAGBench dataset"},
#                                         embedding_function=emberding_function)
# for i, text in enumerate(texts):
#     collection.add(
#         ids=[f"doc_{i}"],
#         documents=[text],
#         metadatas=[{"source": f"legal_doc_{i}"}]
#     )
# print(f"Added {len(texts)} documents to ChromaDB collection 'test'")

Added 200 documents to ChromaDB collection 'test'


In [ ]:
import chromadb
from chromadb.utils import embedding_functions

emberding_function = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="BAAI/bge-small-en-v1.5") #"all-MiniLM-L6-v2"

client = chromadb.PersistentClient(path="./chroma_db1")
collection = client.get_or_create_collection(name="test",
                                        metadata={"description": "Collection of legal documents from RAGBench dataset"},
                                        embedding_function=emberding_function)

for idx, doc in enumerate(raw_documents):
    preprocessed_doc = preprocess_documents(doc)
    chunks = text_splitter.split_text(preprocessed_doc)
    print(f"Document {idx} split into {len(chunks)} chunks.")
    for i, chunk in enumerate(chunks):
        collection.add(
            ids=[f"legal_doc_{idx}_{i}"],
            documents=[chunk],
            metadatas=[{"source": f"legal_doc_{idx}_{i}"}]
        )
print(f"Added {len(raw_documents)} documents to ChromaDB collection 'test'")

Document 0 split into 81 chunks.
Document 1 split into 54 chunks.
Document 2 split into 65 chunks.
Document 3 split into 85 chunks.
Document 4 split into 96 chunks.
Document 5 split into 77 chunks.
Document 6 split into 206 chunks.
Document 7 split into 63 chunks.
Document 8 split into 316 chunks.
Document 9 split into 44 chunks.
Document 10 split into 362 chunks.
Document 11 split into 81 chunks.
Document 12 split into 198 chunks.
Document 13 split into 57 chunks.
Document 14 split into 438 chunks.
Document 15 split into 22 chunks.
Document 16 split into 76 chunks.
Document 17 split into 250 chunks.
Document 18 split into 126 chunks.
Document 19 split into 17 chunks.
Document 20 split into 88 chunks.
Document 21 split into 5 chunks.
Document 22 split into 7 chunks.
Document 23 split into 619 chunks.
Document 24 split into 35 chunks.
Document 25 split into 119 chunks.
Document 26 split into 162 chunks.
Document 27 split into 137 chunks.
Document 28 split into 29 chunks.
Document 29 spl

In [99]:
## List of collections in the ChromaDB client to verify the collection was created and documents were added
# import chromadb
client = chromadb.PersistentClient(path="/Users/chetansunkara/iiith/Capstone Project RAG/rag-pipeline-iiith-aiml-group-11-main/src/vectordb/chroma_db")
client.list_collections()

[]

In [98]:
# # delete the collection after verification to clean up the database
client.delete_collection(name="RagBench")

In [75]:
query = "What are the key legal principles discussed in the document?"
results = collection.query(query_texts=[query],
                 n_results=10)
print(results)

NotFoundError: Error getting collection: Collection [ef8eda05-964d-4a16-bd48-26fda4ddf372] does not exist.

In [74]:
results

{'ids': [['legal_doc_26_152',
   'legal_doc_17_72',
   'legal_doc_26_59',
   'legal_doc_26_40',
   'legal_doc_26_133',
   'legal_doc_17_163',
   'legal_doc_26_143',
   'legal_doc_26_115',
   'legal_doc_10_122',
   'legal_doc_51_131']],
 'embeddings': None,
 'documents': [['-\n1\n7\n-',
   '-\n7\n-',
   '-\n7\n-',
   '-\n5\n-',
   '-\n1\n5\n-',
   '-\n1\n6\n-',
   '-\n1\n6\n-',
   '-\n1\n3\n-',
   '1\n3',
   '1\n3']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'source': 'legal_doc_26_152'},
   {'source': 'legal_doc_17_72'},
   {'source': 'legal_doc_26_59'},
   {'source': 'legal_doc_26_40'},
   {'source': 'legal_doc_26_133'},
   {'source': 'legal_doc_17_163'},
   {'source': 'legal_doc_26_143'},
   {'source': 'legal_doc_26_115'},
   {'source': 'legal_doc_10_122'},
   {'source': 'legal_doc_51_131'}]],
 'distances': [[0.4341864585876465,
   0.4398197531700134,
   0.4398197531700134,
   0.4496386647224426,
   0.45066118240356445,
   0

In [14]:
from sentence_transformers import CrossEncoder
reranker = CrossEncoder("BAAI/bge-reranker-base") #"cross-encoder/ms-marco-MiniLM-L-6-v2"

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 8100.54it/s]


In [49]:
docs = results['documents'][0]
ids = results['ids'][0]
pairs = [[query, doc] for doc in docs]
scores = reranker.predict(pairs)
ranked = sorted(
    zip(scores, ids, docs),
    reverse=True
)
top_docs = [doc for score, id, doc in ranked[:5]]
[print(id,score) for score, id, doc in ranked[:5]]
top_docs

legal_doc_51_131 0.0028766147
legal_doc_10_122 0.0028766147
legal_doc_26_59 0.00015210829
legal_doc_17_72 0.00015210829
legal_doc_26_40 0.0001183705


['1\n3', '1\n3', '-\n7\n-', '-\n7\n-', '-\n5\n-']

In [50]:
top_docs

['1\n3', '1\n3', '-\n7\n-', '-\n7\n-', '-\n5\n-']

In [63]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain_core.output_parsers import JsonOutputParser

class QueryClassification(BaseModel):
    domain: Literal[
        "legal",
        "customer_support",
        "finance",
        "biological",
        "general_knowledge"
    ] = Field(description="Top-level domain")

    dataset: Literal[
        "cuad",
        "delusionqa",
        "emanual",
        "techqa",
        "finqa",
        "tatqa",
        "pubmedqa",
        "covidqarag",
        "hotpotqa",
        "msmarco",
        "hagrid",
        "experqa"
    ] = Field(description="RAGBench dataset")

parser = JsonOutputParser(
pydantic_object=QueryClassification)

In [53]:
from langchain_ollama import ChatOllama
llm = ChatOllama(model="llama3.2:3b")
structured_llm = llm.with_structured_output(QueryClassification)

In [69]:
from langchain_core.prompts import PromptTemplate
CLASSIFICATION_PROMPT = PromptTemplate(template="""
You are an expert query classifier.

Classify the query into ONE of the following RAGBench datasets.

Domains and datasets:

LEGAL
- cuad
  Legal contracts, clauses, obligations, agreements,
  compliance, rights, responsibilities.

CUSTOMER_SUPPORT
- delusionqa
  Customer support conversations and helpdesk issues.

- emanual
  Product manuals, user guides, operating instructions.

- techqa
  Technical troubleshooting, software, hardware,
  IT support.

FINANCE
- finqa
  Financial reasoning, calculations, earnings,
  revenue analysis.

- tatqa
  Financial tables, accounting tables,
  balance sheets, income statements.

BIOLOGICAL
- pubmedqa
  Biomedical research, medicine,
  scientific papers.

- covidqarag
  COVID-19, epidemiology, pandemic,
  medical research.

GENERAL_KNOWLEDGE
- hotpotqa
  Multi-hop reasoning requiring multiple facts.

- msmarco
  Search-engine style question answering.

- hagrid
  Information-seeking questions requiring
  comprehensive answers.

- experqa
  Expert-level open-domain questions.

Return JSON only.

{format_instructions}

Query:
{query}
""",input_variables=["query"],partial_variables={"format_instructions": parser.get_format_instructions()})


In [70]:
classification_chain = (
    CLASSIFICATION_PROMPT
    | llm
    | parser
)

In [71]:
result = classification_chain.invoke(
    {"query": query}
)

print(result)

{'domain': 'legal', 'dataset': 'cuad'}


In [ ]:
# pip install -U langchain langchain-ollama
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 6. Define the Prompt Template
template = """You are an assistant for question-answering tasks. 
Use the following pieces of retrieved context to answer the question. 
If you don't know the answer, just say that you don't know. 

Context:
{context}

Question:
{question}

Answer:"""
prompt = ChatPromptTemplate.from_template(template)

# 7. Define the LLM
llm = ChatOllama(model="gpt-oss:20b", temperature=0)

# Helper function to format retrieved chunks 
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 8. Assemble the LCEL (LangChain Expression Language) RAG Chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 9. Test the Pipeline with a question directly from the dataset
test_query = dataset[0]['question']
print(f"\n--- Running RAG Pipeline ---")
print(f"Query: {test_query}")

response = rag_chain.invoke(test_query)

print("\n--- Generated Answer ---")
print(response)

print("\n--- Original Ground Truth Answer from Dataset ---")
print(dataset[0]['response'])

In [ ]:
legal_data = load_ragbench_subsets(subsets=domain_legal_list, split="test", add_dataset_name=False)
raw_documents = []
seen_docs = set()

# for i in range(min(5, len(legal_data))):  # Indexing a small subset for demonstration
for doc_text in legal_data['documents']:
    for docs in doc_text:
        if docs not in seen_docs:
            seen_docs.add(docs)
            # Wrap standard strings into LangChain Document objects
            raw_documents.append(docs)


print(len(raw_documents))

In [ ]:
import chromadb
from chromadb.utils import embedding_functions

emberding_function = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="BAAI/bge-small-en-v1.5") #"all-MiniLM-L6-v2"

client = chromadb.PersistentClient(path="./chroma_db1")
collection = client.get_or_create_collection(name="test",
                                        metadata={"description": "Collection of legal documents from RAGBench dataset"},
                                        embedding_function=emberding_function)

for idx, doc in enumerate(raw_documents):
    preprocessed_doc = preprocess_documents(doc)
    chunks = text_splitter.split_text(preprocessed_doc)
    print(f"Document {idx} split into {len(chunks)} chunks.")
    for i, chunk in enumerate(chunks):
        collection.add(
            ids=[f"legal_doc_{idx}_{i}"],
            documents=[chunk],
            metadatas=[{"source": f"legal_doc_{idx}_{i}"}]
        )
print(f"Added {len(raw_documents)} documents to ChromaDB collection 'test'")

In [76]:
{"legal": ["cuad"],

        "finance": [
            "finqa",
            "tatqa"
        ],

        "biological": [
            "pubmedqa",
            "covidqa"
        ],

        "customer_support": [
            "delusionqa",
            "emanual",
            "techqa"
        ],

        "general_knowledge": [
            "hotpotqa",
            "msmarco",
            "hagrid",
            "expertqa"
        ]}.items()

dict_items([('legal', ['cuad']), ('finance', ['finqa', 'tatqa']), ('biological', ['pubmedqa', 'covidqa']), ('customer_support', ['delusionqa', 'emanual', 'techqa']), ('general_knowledge', ['hotpotqa', 'msmarco', 'hagrid', 'expertqa'])])